### [  모델 성능 향상을 위한 모델 구성 ] 

- 학습/연산 관련 층 
    * nn.Linear()  - 전결합층 FC Layer.  가중합 연산 수행

- 과대적합 개선을 위한 층
    * nn.Dropout() - 전층으로부터 전달 받은 값을 지정된 확률 비율로 0으로 변경
    * 조합
        - 은닉층 -> AF -> Dropout : 일부를 무작위로 꺼서 규제. 정보 손실이 의도한 대로 정확한 조합 ★
        - 은닉층 -> Dropout -> AF : 양수 값을 0으로 만들 수 있고, 일부 Dropout 효과가 ReLU와 겹쳐질 수 있음
        - 입력층 -> Dropout -> 은닉층 : 어떤 입력 feature가 없어도 모델이 강건하게 동작하게 만드는 목적. 

>> **[1] 모듈 로딩** 

In [6]:
## 모듈 로딩
import torch
import torch.nn as nn
import torch.nn.functional as F

## 재현성 설정 : 전역 난수 시드 
torch.manual_seed(0)

>> **[1] 일반 DNN 모델**

In [7]:
## ---------------------------------------------------------------------------------------------
## 클래스이름 : MNISTDNN
## 클래스기능 : 손글씨 이미지 0~9 분류 모델
## 모델층구성 :     입력         출력        퍼셉트론수      연산여부        활성함수    
##  입력층     이미지픽셀 784    784             X             X             X

##  은닉층 Linear   784          350           350            O           ReLU
##  은닉층 Dropout  350          350                          X             X
##  출력층 Linear   350          10             10            O           다중분류 X
## ---------------------------------------------------------------------------------------------

In [8]:
## ===========================================================
## Dropout 층 순서  [일반] 은닉층 -> AF -> Dropout
## ===========================================================
class MNISTDNN(nn.Module):
    ## -----------------------
    ## 모델 층 구성 및 초기화 
    ## -----------------------
    def __init__(self):
        super().__init__()
        self.hd_layer   = nn.Linear(6, 7)  ## 입력층 -> 은닉층
        self.dropout    = nn.Dropout()      ## 은닉층 -> 은닉층
        self.out_layer  = nn.Linear(7, 4)   ## 은닉층 -> 출력층

    ## -----------------------
    ## 순전파 진행
    ## -----------------------
    def forward(self, data, verbose=False):
        ## 입력층 피쳐 -> 은닉층 가중합 연산 
        weightedsum = self.hd_layer(data)
        print(f'[{"hd_layer":8}] {weightedsum.tolist()}')

        ## 은닉층 가중합연산 -> 활성화 함수
        af_out = F.relu(weightedsum)
        print(f'[{"relu":8}] {af_out.tolist()}')

        ## 전 층 출력값 -> [기] 0.5에 해당하는 출력값 0으로 설정
        ## 다음 층으로 전달
        out = self.dropout(af_out)
        print(f'[{"dropout":8}] {out.tolist()}')

        ## 전 층 출력값 -> 출력층 가중합 연산 수행
        final = self.out_layer(out)
        print(f'[{"out_layer":8}] {final.tolist()}')

        return final

In [9]:
torch.manual_seed(0)
model = MNISTDNN()
x = torch.rand(1, 6)
print("입력값:", x.round(decimals=2).tolist())

print("\n[train 모드] =====")
model.train()
for idx in range(3):
    print(f"\n--- {idx+1}번째 forward ---")
    model(x, verbose=True)

print("\n[eval 모드] =====") 
model.eval()
for idx in range(3):
    print(f"\n--- {idx+1}번째 forward ---")
    model(x, verbose=True)

입력값: [[0.9399999976158142, 0.8799999952316284, 0.0, 0.5899999737739563, 0.41999998688697815, 0.41999998688697815]]

[train 모드] =====

--- 1번째 forward ---
[hd_layer] [[0.06054346263408661, -0.12464538216590881, -0.7230441570281982, -0.33418571949005127, -0.14090749621391296, 0.009384989738464355, -0.05802609771490097]]
[relu    ] [[0.06054346263408661, 0.0, 0.0, 0.0, 0.0, 0.009384989738464355, 0.0]]
[dropout ] [[0.12108692526817322, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]
[out_layer] [[-0.3303227126598358, 0.04329821467399597, -0.22263018786907196, -0.3875143229961395]]

--- 2번째 forward ---
[hd_layer] [[0.06054346263408661, -0.12464538216590881, -0.7230441570281982, -0.33418571949005127, -0.14090749621391296, 0.009384989738464355, -0.05802609771490097]]
[relu    ] [[0.06054346263408661, 0.0, 0.0, 0.0, 0.0, 0.009384989738464355, 0.0]]
[dropout ] [[0.0, 0.0, 0.0, 0.0, 0.0, 0.01876997947692871, 0.0]]
[out_layer] [[-0.32391464710235596, 0.03762297332286835, -0.2320553958415985, -0.3459857702255249]]

In [10]:
from torchinfo import summary 

## 모델 인스턴스 생성
model = MNISTDNN()

## input_size=(BS, 모델 입력 형태)
summary(model, input_size=(1, 6))

[hd_layer] [[-0.2665598392486572, 0.7113369703292847, 0.3345218896865845, -0.9923605918884277, 0.12737302482128143, -0.5582836866378784, -0.12051679193973541]]
[relu    ] [[0.0, 0.7113369703292847, 0.3345218896865845, 0.0, 0.12737302482128143, 0.0, 0.0]]
[dropout ] [[0.0, 0.7113369703292847, 0.3345218896865845, 0.0, 0.12737302482128143, 0.0, 0.0]]
[out_layer] [[-0.3051830530166626, 0.2536499798297882, 0.03148301690816879, 0.10762464255094528]]


Layer (type:depth-idx)                   Output Shape              Param #
MNISTDNN                                 [1, 4]                    --
├─Linear: 1-1                            [1, 7]                    49
├─Dropout: 1-2                           [1, 7]                    --
├─Linear: 1-3                            [1, 4]                    32
Total params: 81
Trainable params: 81
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [11]:
## -------------------------------------------------------------
## F.dropout() 함수
## -------------------------------------------------------------
class MNISTDNN(nn.Module):
    ## -----------------------
    ## 모델 층 구성 및 초기화 
    ## -----------------------
    def __init__(self):
        super().__init__()
        self.hd_layer   = nn.Linear(784, 350)  ## 입력층 -> 은닉층
        self.out_layer  = nn.Linear(350, 10)   ## 은닉층 -> 출력층

    ## -----------------------
    ## 순전파 진행
    ## -----------------------
    def forward(self, data):
        ## 입력층 피쳐 -> 은닉층 가중합 연산 
        weightedsum = self.hd_layer(data)

        ## 은닉층 가중합연산 -> 활성화 함수
        af_out = F.relu(weightedsum)

        ## 전 층 출력값 -> [기] 0.5에 해당하는 출력값 0으로 설정
        ## 다음 층으로 전달
        out = F.dropout(af_out)

        ## 전 층 출력값 -> 출력층 가중합 연산 수행
        final = self.out_layer(out)

        return final

In [12]:
from torchinfo import summary 

## 모델 인스턴스 생성
model = MNISTDNN()

## input_size=(BS, 모델 입력 형태)
summary(model, input_size=(1, 784))

Layer (type:depth-idx)                   Output Shape              Param #
MNISTDNN                                 [1, 10]                   --
├─Linear: 1-1                            [1, 350]                  274,750
├─Linear: 1-2                            [1, 10]                   3,510
Total params: 278,260
Trainable params: 278,260
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.28
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 1.11
Estimated Total Size (MB): 1.12